# HealthConnect Clinic — Initial Analysis Notebook
### AnalystLab Africa Experience Lab — Week 4
**Track:** Data Analytics
**Prepared by:** Jeffrey Uwoghiren
**Date:** 28 August 2026

This notebook is the Week 4 Initial Analysis Document for the Data Analytics track of the
HealthConnect Experience Lab. It covers, in the order required by the Week 4 assignment brief:

1. Dataset Overview
2. Data Quality Assessment
3. Identification of Important Variables
4. Business Questions
5. Proposed KPIs (justified, not yet calculated — per brief instructions)
6. Initial Analysis Approach
7. Assumptions, Limitations, Risks and Dependencies

**Resource note:** This notebook reads `HealthConnect_Appointment_Data.csv` and
`HealthConnect_Data_Dictionary - Data Dictionary.csv` directly from the project root.
The original files are never modified — this notebook is read-only against them.


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('../data/HealthConnect_Appointment_Data.csv')
data_dict = pd.read_csv('../data/HealthConnect_Data_Dictionary - Data Dictionary.csv')

print(f"Appointment records: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


Appointment records: 5,000
Columns: 18


## 1. Dataset Overview

The dataset is a single, appointment-level table — one row per appointment, with no joins
required. It contains patient demographics, appointment scheduling details, booking
behaviour, prior attendance history, reminder information, distance to clinic, and the
final appointment outcome.


In [2]:
data_dict

,Variable,Data Type,Description,Example,Notes
0,appointment_id,Text,Unique appointment identifier,HC-00001,Primary key
1,patient_id,Text,Anonymised patient identifier,P-0421,May appear across multiple appointments
2,gender,Text,Recorded gender category,Female,Synthetic and anonymised
3,age,Integer,Patient age in years,34,Adult patients only
4,age_group,Text,Age band derived from age,25-34,Provided for descriptive analysis
5,appointment_type,Text,Type of scheduled appointment,Follow-up,Four appointment categories
6,booking_date,Date,Date appointment was booked,2025-04-10,ISO format
7,appointment_date,Date,Scheduled appointment date,2025-04-24,ISO format
8,appointment_day,Text,Day of week for appointment,Thursday,Derived from appointment date
9,appointment_time,Text,Appointment time period,Morning,"Morning, Afternoon, Evening"


In [3]:
df.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [4]:
df.dtypes

appointment_id               str
patient_id                   str
gender                       str
age                        int64
age_group                    str
appointment_type             str
booking_date                 str
appointment_date             str
appointment_day              str
appointment_time             str
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent                str
reminder_channel             str
distance_to_clinic_km    float64
waiting_time_minutes     float64
appointment_outcome          str
dtype: object

## 2. Data Quality Assessment

Each check below is run directly against the data rather than assumed, so the quality
verdict is evidence-based.


In [5]:
print("Duplicate appointment_id:", df['appointment_id'].duplicated().sum())
print("Fully duplicate rows:", df.duplicated().sum())


Duplicate appointment_id: 0
Fully duplicate rows: 0


In [6]:
print("Null counts by column:")
df.isna().sum()


Null counts by column:


appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_time            0
booking_lead_days           0
previous_appointments       0
previous_no_shows           0
reminder_sent               0
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
appointment_outcome         0
dtype: int64

**Interpretation of `reminder_channel` nulls:** every null in `reminder_channel` should
correspond exactly to `reminder_sent == 'No'` — i.e. the field is structurally absent, not
genuinely missing data, if a reminder was never sent.


In [7]:
mismatch = df[(df['reminder_channel'].isna()) & (df['reminder_sent'] == 'Yes')]
print(f"Rows where reminder_channel is null but reminder_sent == 'Yes': {len(mismatch)}")
print("=> confirms reminder_channel nulls are structural, not a data quality issue.")


Rows where reminder_channel is null but reminder_sent == 'Yes': 0
=> confirms reminder_channel nulls are structural, not a data quality issue.


**Genuinely missing fields:** `distance_to_clinic_km` and `waiting_time_minutes` have a
small number of true nulls (no structural explanation). These are flagged for a documented
handling decision — see Assumptions & Limitations (Section 7).


In [8]:
for col in ['distance_to_clinic_km', 'waiting_time_minutes']:
    n = df[col].isna().sum()
    pct = n / len(df) * 100
    print(f"{col}: {n} missing ({pct:.1f}%)")


distance_to_clinic_km: 90 missing (1.8%)
waiting_time_minutes: 60 missing (1.2%)


In [9]:
# Logical consistency checks
violations_no_shows = (df['previous_no_shows'] > df['previous_appointments']).sum()
violations_dates = (pd.to_datetime(df['booking_date']) > pd.to_datetime(df['appointment_date'])).sum()

print("previous_no_shows > previous_appointments (should be 0):", violations_no_shows)
print("booking_date after appointment_date (should be 0):", violations_dates)


previous_no_shows > previous_appointments (should be 0): 0
booking_date after appointment_date (should be 0): 0


In [10]:
df['appointment_outcome'].value_counts()

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

**Verdict:** the dataset is clean and internally consistent — no duplicates, no logical
violations, and the two genuinely-missing fields affect under 2% of records each. The
three-class outcome (`Attended` / `No-Show` / `Cancelled`) is roughly balanced between
Attended and No-Show, with Cancelled as a smaller, distinct third category that must not
be merged into No-Show (see Section 3).


## 3. Identification of Important Variables

A variable is only "important" for explaining appointment attendance if the No-Show rate
**actually changes** as that variable changes. If the rate stays flat across all values of a
variable, that variable carries no discriminating information relative to the overall base
rate. Each candidate variable is tested against this principle below, using only
`Attended` and `No-Show` records (Cancelled excluded, since it is a separate outcome —
see Section 2).


In [11]:
outcomes = df[df['appointment_outcome'].isin(['Attended', 'No-Show'])].copy()
base_rate = (outcomes['appointment_outcome'] == 'No-Show').mean()
print(f"Overall No-Show rate (Attended vs No-Show only): {base_rate:.1%}")


Overall No-Show rate (Attended vs No-Show only): 51.2%


In [12]:
def no_show_rate_by(col, bins=None, labels=None):
    d = outcomes.copy()
    if bins is not None:
        d[col] = pd.cut(d[col], bins=bins)
    rate = d.groupby(col, observed=True)['appointment_outcome'].apply(lambda x: (x == 'No-Show').mean())
    return rate.round(3)


In [13]:
print("No-Show rate by previous_no_shows:")
print(no_show_rate_by('previous_no_shows'))


No-Show rate by previous_no_shows:
previous_no_shows
0    0.463
1    0.559
2    0.621
3    0.697
4    0.667
5    1.000
Name: appointment_outcome, dtype: float64


In [14]:
print("No-Show rate by booking_lead_days (banded):")
print(no_show_rate_by('booking_lead_days', bins=[-1, 3, 7, 14, 30, 60, 200]))


No-Show rate by booking_lead_days (banded):


booking_lead_days
(-1, 3]     0.266
(3, 7]      0.322
(7, 14]     0.352
(14, 30]    0.455
(30, 60]    0.639
Name: appointment_outcome, dtype: float64

In [15]:
print("No-Show rate by distance_to_clinic_km (banded):")
print(no_show_rate_by('distance_to_clinic_km', bins=[0, 5, 10, 20, 30, 50]))


No-Show rate by distance_to_clinic_km (banded):
distance_to_clinic_km
(0, 5]      0.487
(5, 10]     0.493
(10, 20]    0.523
(20, 30]    0.584
(30, 50]    0.700
Name: appointment_outcome, dtype: float64


In [16]:
print("No-Show rate by reminder_sent:")
print(no_show_rate_by('reminder_sent'))


No-Show rate by reminder_sent:
reminder_sent
No     0.546
Yes    0.499
Name: appointment_outcome, dtype: float64


In [17]:
print("No-Show rate by reminder_channel, waiting_time_minutes (banded), appointment_type, "
      "appointment_time, appointment_day, age_group, gender — checked for completeness:\n")

print("reminder_channel:")
print(no_show_rate_by('reminder_channel'))
print("\nwaiting_time_minutes (banded):")
print(no_show_rate_by('waiting_time_minutes', bins=[0, 15, 30, 45, 70]))
print("\nappointment_type:")
print(no_show_rate_by('appointment_type'))
print("\nappointment_time:")
print(no_show_rate_by('appointment_time'))
print("\nappointment_day:")
print(no_show_rate_by('appointment_day'))
print("\nage_group:")
print(no_show_rate_by('age_group'))
print("\ngender:")
print(no_show_rate_by('gender'))


No-Show rate by reminder_channel, waiting_time_minutes (banded), appointment_type, appointment_time, appointment_day, age_group, gender — checked for completeness:

reminder_channel:
reminder_channel
Email       0.510
SMS         0.480
WhatsApp    0.527
Name: appointment_outcome, dtype: float64

waiting_time_minutes (banded):


waiting_time_minutes
(0, 15]     0.499
(15, 30]    0.522
(30, 45]    0.497
(45, 70]    0.500
Name: appointment_outcome, dtype: float64

appointment_type:
appointment_type
Diagnostic Test            0.520
Follow-up                  0.542
General Consultation       0.491
Specialist Consultation    0.505
Name: appointment_outcome, dtype: float64

appointment_time:
appointment_time
Afternoon    0.512
Evening      0.526
Morning      0.507
Name: appointment_outcome, dtype: float64

appointment_day:
appointment_day
Friday       0.487
Monday       0.531
Saturday     0.496
Sunday       0.528
Thursday     0.520
Tuesday      0.489
Wednesday    0.528
Name: appointment_outcome, dtype: float64

age_group:


age_group
18-24    0.524
25-34    0.529
35-44    0.514
45-54    0.511
55-64    0.530
65+      0.481
Name: appointment_outcome, dtype: float64

gender:
gender
Female               0.511
Male                 0.515
Prefer not to say    0.448
Name: appointment_outcome, dtype: float64


### Conclusion — Important Variables

**Strong signal (retained):**
- `previous_no_shows` (and `previous_appointments` as context/denominator) — No-Show rate
  climbs from ~46% at zero prior no-shows to 60–70%+ at 2–4 prior no-shows. The single
  strongest variable in the dataset.
- `booking_lead_days` — No-Show rate rises from ~27% (booked within 3 days) to ~64%
  (booked 30–60 days out).
- `distance_to_clinic_km` — No-Show rate rises from ~49% (under 5km) to ~70% (30–50km).

**Weak signal (retained for its operational relevance despite a modest effect):**
- `reminder_sent` — a small gap (~55% vs ~50%), directionally as expected but far weaker
  than the business scenario's framing implies.

**No meaningful signal (deprioritised):**
- `reminder_channel`, `waiting_time_minutes`, `appointment_type`, `appointment_time`,
  `appointment_day`, `age_group`, `gender` — all sit within a narrow band close to the
  overall base rate, with no meaningful separation.

**Why the flat variables were deprioritised, not just excluded:** each represents either an
operational/logistics detail unrelated to patient intent or ability to attend
(`reminder_channel`, scheduling variables), or a demographic proxy that does not, in this
dataset, correlate with the outcome (`age_group`, `gender`). Ruling these out is itself a
useful, evidence-based finding — it tells the clinic not to prioritise reminder-channel or
scheduling changes as no-show interventions.


## 4. Business Questions

Each question below is grounded in a variable validated in Section 3, and each is framed to
be actionable — answerable in a way that could inform a clinic decision, not just descriptive
for its own sake.

1. **How does a patient's prior no-show history predict the likelihood of missing their
   next appointment?** *(`previous_no_shows` / `previous_appointments`)*
2. **Does booking lead time affect the likelihood of a no-show, and is there a threshold
   where risk sharply increases?** *(`booking_lead_days`)*
3. **Does distance to the clinic affect attendance, and are certain distance bands
   disproportionately at risk?** *(`distance_to_clinic_km`)*
4. **Does sending a reminder measurably reduce no-shows, and is the current reminder
   strategy effective?** *(`reminder_sent`)*
5. **Should cancellations be treated separately from no-shows in how the clinic measures
   and responds to missed slots?** *(`appointment_outcome`, three-class structure)*


## 5. Proposed KPIs

Per the Week 4 brief, KPIs are **identified and justified only** at this stage — not
calculated, analysed, or visualised. Each KPI below is a rate broken out by a validated
variable and is explicitly linked to one of the five business questions above, so that
every question has a trackable metric.

| # | KPI | Linked Question | Justification |
|---|-----|-----------------|----------------|
| 1 | **No-Show Rate by Prior No-Show Bracket** | Q1 | Tracks whether risk-tiering patients by history is predictive over time, and whether interventions for high-risk patients move the rate. |
| 2 | **No-Show Rate by Booking Lead-Time Band** | Q2 | Directly actionable — if the clinic caps advance booking or adds reconfirmation steps, this KPI shows whether it worked. |
| 3 | **No-Show Rate by Distance Band** | Q3 | Tracks whether distance-based interventions (transport support, telehealth) close the gap between near and far patients. |
| 4 | **Wasted Slot Rate** (No-Show count / total scheduled appointments), cut by reminder status | Q4 and the central project question | The umbrella KPI — answers the scenario's stated cost ("inefficient use of appointment slots") directly, and the reminder-status cut answers Q4 without requiring a separate KPI. |
| 5 | **Cancellation-to-No-Show Ratio** | Q5 | Tracks whether missed slots are increasingly being converted from silent no-shows into proper cancellations — a rising ratio is a success signal even before attendance itself improves. |

**Why exactly these five:** the KPI set intentionally excludes a standalone "Reminder
Effectiveness Rate" and a "composite risk score." The former is folded into KPI 4 (its
signal is too weak on its own to justify a dedicated KPI). The latter would require
model-style weighting logic that belongs to the Data Science track's scope, not Data
Analytics — including it here would blur the track boundary defined in the assignment brief.


## 6. Initial Analysis Approach

**1. Data preparation.** Resolve the two genuine missing-value gaps (`distance_to_clinic_km`,
`waiting_time_minutes`) before any KPI calculation. Since `waiting_time_minutes` was
deprioritised in Section 3, its missingness is low-priority. For `distance_to_clinic_km`
(relevant to KPI 3), the recommended approach is to exclude the ~1.8% of affected rows from
distance-based analysis rather than impute, to avoid introducing synthetic bias into a
genuinely predictive variable. `appointment_outcome` is kept as three distinct classes
throughout — Cancelled is never merged into No-Show. Any cleaned/derived dataset is saved
separately; the original CSV is never modified.

**2. Segment and compute each KPI.** For each of the five KPIs, group by the relevant
validated variable and compute the rate, in order of evidentiary strength established in
Section 3: prior no-shows first, then lead time, then distance, then the reminder-status
cut on Wasted Slot Rate, then the cancellation ratio.

**3. Validate findings aren't spurious.** Check that segment/bin sample sizes are large
enough to be reliable (e.g. the 100% No-Show rate at `previous_no_shows = 5` is based on a
small number of rows and should not be over-interpreted). Lightly check whether patterns
hold when two variables are combined (e.g. does the lead-time effect persist within the
high-distance group, or is it confounded?) — this is a sanity check, not full modelling.

**4. Tooling.** Python (pandas) for computation and data-quality handling, consistent with
this notebook. Power BI or Tableau for stakeholder-facing visuals, once this analysis moves
beyond Week 4's scope of identification and justification.

**5. Output.** All of the above feeds into this notebook (the Initial Analysis Document)
and the separate Week 4 Project Summary.


## 7. Assumptions, Limitations, Risks and Dependencies

- **Data limitations.** `distance_to_clinic_km` and `waiting_time_minutes` have small
  amounts of genuine missing data (~1.8% / ~1.2%). Any distance-based KPI should be
  understood as computed on ~98% of appointments, not all of them.
- **Synthetic data.** Booking and appointment dates extend into 2026, confirming this is
  simulated data for training purposes, not live clinic records. Patterns found here are
  valid *within this dataset*, not validated against real-world behaviour.
- **Small-sample artifacts.** Extreme rates at the tails of a distribution (e.g.
  `previous_no_shows = 5`) are likely based on very few rows and should not be treated as a
  hard rule without checking segment sizes.
- **Correlation, not causation.** All findings in Section 3 are descriptive associations,
  not causal claims — no confounding has been controlled for (e.g. whether long-lead-time
  bookings also tend to be far-distance patients). Causal/predictive modelling is explicitly
  the Data Science track's responsibility, not this analysis's.
- **Cross-track dependency.** These KPIs and questions are the foundation the Data Science
  track will build a predictive model on. If they define the target variable differently
  (e.g. collapsing Cancelled into No-Show, which this analysis argues against), the KPI
  framing here and the DS track's modelling target could diverge — this needs explicit
  alignment before DS modelling begins.
- **Reminder effectiveness caveat.** Reminders show only a weak effect in this dataset,
  which runs counter to the assumption embedded in the business scenario. This is stated
  plainly as an evidence-based finding, not softened to match the scenario's framing.
